# 🤖 Phase 2 — Week 3 & 4: Logistic Regression + Spam Post Detector

---

**What you'll learn:**
- Classification vs Regression — the key difference
- How Logistic Regression works (intuitively)
- New evaluation metrics: Accuracy, Precision, Recall, F1-score
- Confusion Matrix — what it is and how to read it
- Class imbalance — a real-world problem you'll always face
- Mini Project: Social Media Spam Post Detector

> 💡 Classification is the #1 most common ML task in real jobs.
> Spam detection, fraud detection, churn prediction — all classification.

---
## 🧠 Section 1: Classification vs Regression

Last week you predicted a **number** (engagement score). That's **Regression.**

This week you'll predict a **category** (spam or not spam). That's **Classification.**

| Task | Output type | Algorithm |
|---|---|---|
| Predict engagement score | Number (1200, 4500, 890) | Linear Regression |
| Predict is post spam? | Category (yes / no) | Logistic Regression |
| Predict post category | Category (tech/news/sports) | Logistic Regression / others |

---

## 📐 Section 2: How Logistic Regression Works — Intuitively

Despite the name, Logistic Regression is a **classification** algorithm, not regression.

It answers: **"What is the probability this post is spam?"**

Internally it does this:
1. Same as Linear Regression — calculates a weighted sum of features
2. Then squashes the result into a probability between 0 and 1 using the **sigmoid function**
3. If probability > 0.5 → predict class 1 (spam)
4. If probability < 0.5 → predict class 0 (not spam)

```
Linear Regression:  output = 4500  (a number)
Logistic Regression: output = 0.87  (87% chance it's spam → predict spam)
```

That's the only real difference. The learning process is almost identical.

In [ ]:
# ✅ Visualize the sigmoid function — so you can SEE what Logistic Regression does

import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.linspace(-10, 10, 200)
y = sigmoid(x)

plt.figure(figsize=(8, 4))
plt.plot(x, y, color="#6366f1", linewidth=2)
plt.axhline(y=0.5, color="red", linestyle="--", label="Decision boundary (0.5)")
plt.axhline(y=1.0, color="gray", linestyle=":", alpha=0.5)
plt.axhline(y=0.0, color="gray", linestyle=":", alpha=0.5)
plt.title("Sigmoid Function — squashes any number into 0 to 1")
plt.xlabel("Weighted sum of features")
plt.ylabel("Probability")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Left of 0 → probability < 0.5 → predict NOT spam
# Right of 0 → probability > 0.5 → predict SPAM


---
## 🛠️ Section 3: Build the Dataset — Social Media Spam Posts

**Spam post characteristics (rules we'll bake into the data):**
- Very high like count but very low comments (fake engagement)
- Extremely high share count (bots sharing)
- Very short caption length
- Posted at unusual hours (2am–5am)
- Multiple hashtags

In a real project, these rules come from domain knowledge + data exploration.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

# --- Generate legitimate posts (label = 0) ---
n_legit = 800
legit = pd.DataFrame({
    "likes":          np.random.randint(50, 10000, n_legit),
    "comments":       np.random.randint(10, 1000, n_legit),
    "shares":         np.random.randint(5, 500, n_legit),
    "caption_length": np.random.randint(20, 300, n_legit),
    "hashtag_count":  np.random.randint(1, 10, n_legit),
    "hour_posted":    np.random.randint(6, 23, n_legit),
    "is_spam":        0
})

# --- Generate spam posts (label = 1) ---
n_spam = 200
spam = pd.DataFrame({
    "likes":          np.random.randint(5000, 50000, n_spam),  # inflated likes
    "comments":       np.random.randint(0, 20, n_spam),        # almost no comments
    "shares":         np.random.randint(1000, 5000, n_spam),   # bot shares
    "caption_length": np.random.randint(1, 15, n_spam),        # very short
    "hashtag_count":  np.random.randint(15, 30, n_spam),       # hashtag stuffing
    "hour_posted":    np.random.randint(1, 5, n_spam),         # odd hours
    "is_spam":        1
})

# Combine and shuffle
df = pd.concat([legit, spam], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["is_spam"].value_counts())
print(f"\nSpam rate: {df['is_spam'].mean()*100:.1f}%")
df.head()


In [ ]:
# ✅ Explore differences between spam and legit posts visually

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Spam vs Legitimate Posts — Feature Comparison", fontsize=14, fontweight="bold")

features = ["likes", "comments", "shares", "caption_length", "hashtag_count", "hour_posted"]
colors = {0: "#6366f1", 1: "#ef4444"}
labels = {0: "Legitimate", 1: "Spam"}

for ax, feature in zip(axes.flatten(), features):
    for label in [0, 1]:
        data = df[df["is_spam"] == label][feature]
        ax.hist(data, bins=30, alpha=0.6, color=colors[label], label=labels[label])
    ax.set_title(feature)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 💡 Look at these charts carefully.
# You should see clear separation between spam and legit on most features.
# That separation is what the model will learn to use.


---
## 🏋️ Section 4: Train the Spam Detector

Same 3-step sklearn pattern as Linear Regression.
Only the model class changes — everything else is identical.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Features and label
X = df.drop("is_spam", axis=1)
y = df["is_spam"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ⚠️ NEW STEP: Feature Scaling
# Logistic Regression works better when all features are on the same scale.
# likes can be 50,000 while hour_posted is 1-23 — very different scales.
# StandardScaler transforms each feature to mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only!
X_test_scaled  = scaler.transform(X_test)         # apply same scale to test

# Train
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("✅ Spam detector trained!")


---
## 📏 Section 5: New Evaluation Metrics for Classification

R² and MAE don't work for classification. You need different metrics.

Let's use a real scenario to understand them:
> Out of 100 posts, 10 are spam. Your model has to find them.

| Metric | Question it answers | Formula |
|---|---|---|
| **Accuracy** | Overall, how often was the model right? | correct / total |
| **Precision** | Of all posts predicted spam, how many actually were? | true spam / predicted spam |
| **Recall** | Of all actual spam posts, how many did the model catch? | caught spam / all spam |
| **F1 Score** | Balance between precision and recall | 2 × (P × R) / (P + R) |

**Why Accuracy alone is misleading:**
If 90% of posts are legit, a model that predicts "not spam" for EVERYTHING gets 90% accuracy — but catches zero spam. This is the **class imbalance problem.**

Always look at Precision, Recall, and F1 — not just accuracy.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test_scaled)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 45)
print("       SPAM DETECTOR EVALUATION")
print("=" * 45)
print(f"Accuracy:  {accuracy*100:.1f}%  — overall correct predictions")
print(f"Precision: {precision*100:.1f}%  — of flagged spam, actually spam")
print(f"Recall:    {recall*100:.1f}%  — of all spam, model caught this %")
print(f"F1 Score:  {f1*100:.1f}%  — balance of precision + recall")

print("\n--- Full Report ---")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Spam"]))


In [ ]:
# ✅ Confusion Matrix — the most informative classification visual
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted\nLegit", "Predicted\nSpam"])
ax.set_yticklabels(["Actual\nLegit", "Actual\nSpam"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                fontsize=20, fontweight="bold",
                color="white" if cm[i, j] > cm.max()/2 else "black")

ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.colorbar(im)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (correctly said legit): {tn}")
print(f"False Positives (wrongly flagged as spam): {fp}")
print(f"False Negatives (missed spam): {fn}")
print(f"True Positives  (correctly caught spam): {tp}")


In [ ]:
# ✅ See the probability scores — not just yes/no
# This is what Logistic Regression actually outputs before thresholding

proba = model.predict_proba(X_test_scaled)

results = pd.DataFrame({
    "actual":       y_test.values[:8],
    "predicted":    y_pred[:8],
    "prob_legit":   proba[:8, 0].round(3),
    "prob_spam":    proba[:8, 1].round(3),
})
print(results.to_string(index=False))

# 💡 prob_spam > 0.5 → predicted as spam
# You can adjust this threshold — lower it to catch MORE spam (higher recall)
# but you'll also flag more legit posts (lower precision)
# This trade-off is a real business decision in production systems!


In [ ]:
# ✅ Predict on brand new posts — the real use case

new_posts = pd.DataFrame({
    "likes":          [250,   45000, 1200],
    "comments":       [80,    3,     340],
    "shares":         [30,    2200,  90],
    "caption_length": [180,   5,     120],
    "hashtag_count":  [4,     28,    6],
    "hour_posted":    [14,    3,     19],
})

new_posts_scaled = scaler.transform(new_posts)
predictions = model.predict(new_posts_scaled)
probabilities = model.predict_proba(new_posts_scaled)

post_labels = ["Post A (looks legit)", "Post B (looks spammy)", "Post C (normal)"]

print("=== SPAM DETECTION RESULTS ===")
for label, pred, prob in zip(post_labels, predictions, probabilities):
    result = "🚨 SPAM" if pred == 1 else "✅ LEGIT"
    print(f"{label}")
    print(f"  → {result} (spam probability: {prob[1]*100:.1f}%)")


---
## 🎯 Section 6: Exercises

### Exercise 1
The default decision threshold is 0.5. Change it to **0.3** (more aggressive spam detection).
Use `model.predict_proba()` and manually apply the threshold.
How does Recall change? How does Precision change? Why?

```python
# Hint:
proba = model.predict_proba(X_test_scaled)
y_pred_aggressive = (proba[:, 1] >= 0.3).astype(int)
```

### Exercise 2
Train a second Logistic Regression using **only 3 features**: `comments`, `hashtag_count`, `hour_posted`.
Compare its F1 score to the full model. Which features matter most?

### Exercise 3
Add a new feature `like_to_comment_ratio = likes / (comments + 1)` to the dataset.
Retrain and check if F1 score improves.

> 💡 The `+ 1` in the denominator prevents division by zero when comments = 0.
> This is called **Laplace smoothing** — a real technique used in production.

In [ ]:
# 🏋️ YOUR WORKSPACE

# ✏️ Exercise 1:


# ✏️ Exercise 2:


# ✏️ Exercise 3:



---
## ✅ Week 3–4 Checklist

- [ ] Can explain the difference between classification and regression
- [ ] Understand what the sigmoid function does
- [ ] Know when to use Accuracy vs Precision vs Recall vs F1
- [ ] Can read a Confusion Matrix (TP, TN, FP, FN)
- [ ] Understand why class imbalance makes accuracy misleading
- [ ] Know what StandardScaler does and WHY it's needed
- [ ] Understand the precision/recall trade-off when changing threshold
- [ ] Completed all 3 exercises

---

## 🔜 What's Next — Phase 2, Week 5–6

**Decision Trees + Random Forest**

A completely different type of model that:
- Needs NO feature scaling
- Handles non-linear patterns that Logistic Regression misses
- Is highly interpretable (you can literally visualize the decisions)
- Random Forest = multiple trees combined → much more powerful

Project: **Predict whether a social media post will go viral** (your most complex model yet)

> 💬 Finish the exercises, come back with your solutions and let's keep going!